# DejaVu — Anticipated Scenario Dataset

Carrega o CSV gerado pelo `AntecipatedScenarioDatasetRecorder` e exibe o DataFrame.

---

## Dicionário de colunas (97 colunas)

### 1. Identidade do tick

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `episode` | int | Número do episódio. Cada episódio é uma execução completa da tarefa do zero. |
| `step` | int | Número do tick dentro do episódio. Incrementado a cada percepção recebida do simulador. |

---

### 2. Parâmetros monitorados pelo ASM (State Machine)

Derivados dos sensores brutos pelo `MonitorARM` e usados diretamente como guards/condições na State Machine do DejaVu. Todos são inteiros para compatibilidade com guards booleanos da SM.

| Coluna | Tipo | Como é calculado | Descrição |
|--------|------|-----------------|-----------|
| `task_started` | int (0/1) | Hardcoded = 1 | Tarefa iniciada. Sempre 1 enquanto o episódio corre. |
| `object_available` | int (0/1) | Hardcoded = 1 | Objeto disponível na cena. Sempre 1 neste cenário. |
| `gripper_width_cm` | int | `int(fingers_width * 100)` | Abertura da garra em cm. Aberta ≥ 7.5 cm, fechada ≤ 6 cm (histerese entre 6 e 7.5). |
| `distance_ee_object_cm` | int | `max(0, (dist_ee_to_cube − 0.025) × 100)` | Distância do end-effector ao ponto de contato do cubo em cm. Desconta 2.5 cm do comprimento dos dedos do Franka Panda. |
| `grasp_completed` | int (0/1) | `1 se fingers_width ≤ 0.060 m` | Garra considerada fechada sobre o objeto. Histerese: só reabre se fingers_width > 0.075 m. |
| `finger_contacts` | int (0–2) | Lido direto do simulador | Número de dedos em contato físico com o objeto. |
| `grasp_attempts` | int | Contador acumulativo por episódio | Quantas vezes a garra fechou (transições 0→1 em `grasp_completed`) desde o início do episódio. |
| `object_lift_height_cm` | int | `max(0, (cube_z − cube_z_inicial) × 100)` | Altura do cubo acima da posição inicial em cm. Zero até o cubo ser levantado. |
| `distance_object_goal_cm` | int | `norm(cube_xy − target_xy) × 100` | Distância XY (plano horizontal) entre o cubo e o target em cm. Ignora a componente Z. |
| `task_aborted` | int (0/1) | `1 se current_task == "ABORT" AND is_success` | Tarefa abortada de forma controlada. Normalmente 0. |

---

### 3. Sensores brutos — escalares

Leituras diretas do simulador, antes de qualquer abstração.

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `fingers_width` | float | metros | Abertura real da garra medida entre os dois dedos. |
| `dist_ee_to_cube` | float | metros | Distância 3D do end-effector ao centro geométrico do cubo. |
| `dist_cube_to_target` | float | metros | Distância 3D do cubo à posição do target (inclui Z). |
| `obstacle_in_path` | bool | — | True se algum obstáculo está no caminho do end-effector. |
| `obstacle_count_in_path` | int | — | Número de obstáculos detectados no caminho. |
| `reward` | float | — | Recompensa do agente RL neste tick. Tipicamente negativo e proporcional à distância ao objetivo. |
| `is_success` | bool | — | True se a tarefa foi concluída com sucesso. Sinaliza fim do episódio. |

---

### 4. End-effector — posição e velocidade

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `ee_x`, `ee_y`, `ee_z` | float | metros | Posição 3D do end-effector (ponta da garra) no espaço cartesiano. |
| `ee_vx`, `ee_vy`, `ee_vz` | float | m/s | Velocidade linear 3D do end-effector. |

---

### 5. Cubo — posição, orientação e velocidade

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `cube_x`, `cube_y`, `cube_z` | float | metros | Posição 3D do centro do cubo. |
| `cube_roll`, `cube_pitch`, `cube_yaw` | float | radianos | Orientação do cubo (ângulos de Euler). |
| `cube_vx`, `cube_vy`, `cube_vz` | float | m/s | Velocidade linear do cubo. Não nula quando o cubo está sendo movido. |

---

### 6. Target (objetivo)

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `target_x`, `target_y`, `target_z` | float | metros | Posição 3D da posição-alvo onde o cubo deve ser colocado. Constante durante o episódio. |

---

### 7. Ação do agente RL

Saída da política do agente de Reinforcement Learning neste tick, antes de ser executada.

| Coluna | Tipo | Range | Descrição |
|--------|------|-------|-----------|
| `action_x`, `action_y`, `action_z` | float | [−1, 1] | Deslocamento comandado ao end-effector nos eixos X, Y, Z. |
| `action_gripper` | float | [−1, 1] | Comando da garra: +1 = abrir, −1 = fechar. |

---

### 8. Ângulos das juntas do robô

7 juntas do Franka Panda, da base até o pulso.

| Coluna | Tipo | Unidade | Descrição |
|--------|------|---------|-----------|
| `j0`–`j6` | float | radianos | Ângulo atual de cada junta. |
| `jv0`–`jv6` | float | rad/s | Velocidade angular atual de cada junta. |

---

### 9. Contexto da tarefa

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `current_task` | str | Nome da tarefa de alto nível. Ex: `OBJECT_DELIVERY_SEQUENCE`. |
| `current_subtask` | str | Subtarefa ativa no Managing neste tick. Valores: `APPROACH_OBJECT`, `GRASP_OBJECT`, `RETRY_GRASP`, `SAFE_ABORT`, `LIFT_OBJECT`, `TRANSPORT_OBJECT`, `PLACE_OBJECT`. |
| `active_target_name` | str | Nome do target ativo. Ex: `"target"`. |

---

### 10. Objetos da cena (flattenado)

Propriedades do cubo conforme definido na configuração do episódio. Constantes durante o episódio, exceto `current_position`.

| Coluna | Descrição |
|--------|-----------|
| `objects.object_1.type` | Tipo do objeto. Ex: `"box"`. |
| `objects.object_1.size.0/1/2` | Dimensões XYZ em metros. |
| `objects.object_1.mass` | Massa em kg. |
| `objects.object_1.color.0/1/2/3` | Cor RGBA, cada componente em [0, 1]. |
| `objects.object_1.initial_position.0/1/2` | Posição XYZ inicial do cubo em metros. |
| `objects.object_1.lateral_friction` | Coeficiente de atrito lateral. |
| `objects.object_1.spinning_friction` | Coeficiente de atrito rotacional. |
| `objects.object_1.current_position.0/1/2` | Posição XYZ atual do cubo (maior precisão que `cube_x/y/z`). |

---

### 11. Cena (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `scene.table.length`, `scene.table.width`, `scene.table.height` | Dimensões da mesa em metros. |
| `scene.table.x_offset` | Offset X da mesa em relação à origem. |
| `scene.table.lateral_friction`, `scene.table.spinning_friction` | Atrito da superfície da mesa. |

---

### 12. Configuração do robô (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `robot_config.control_type` | Tipo de controle: `"ee"` (end-effector space) ou `"joint"`. |
| `robot_config.block_gripper` | True se a garra está travada e não responde a comandos. |
| `robot_config.base_position.0/1/2` | Posição XYZ da base do robô em metros. |

---

### 13. Objetivo da tarefa (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `target_goal.type` | Tipo de sequência. Ex: `"goal_sequence"`. |
| `target_goal.targets.0.name` | Nome do target. Ex: `"target"`. |
| `target_goal.targets.0.position.0/1/2` | Posição XYZ do target em metros. |
| `target_goal.mode` | Modo de execução. Ex: `"goal_sequence"`. |

---

### 14. Scripts de comportamento (flattenado)

| Coluna | Descrição |
|--------|-----------|
| `scripts.script_1` | Script de comportamento 1 ativo (bool). |
| `scripts.reach_only` | True se o episódio é de apenas aproximação (sem pegar). |
| `scripts.left_right` | True se o modo de movimento lateral está ativado. |

---

### 15. Monitoramento DejaVu ⭐

Colunas produzidas exclusivamente pelo DejaVu. São o **output do monitoramento** — não existem no Managing nem no simulador.

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `active_state` | str | Estado ativo na State Machine do DejaVu após processar o tick. Segue o fluxo: `INIT → PHI_1 → S2 → PHI_3 → S4 → PHI_5 → S6 → PHI_7 → S8 → ... → FINAL`. Estados `PHI_N` são de verificação (transientes), estados `SN` são estáveis (aguardando próxima ação), `ERR_N` indicam violação, `FINAL` = tarefa concluída. |
| `sat` | bool / None | **Label do dataset.** `True` = comportamento confirmado como antecipado (SM avançou sem erro). `False` = cenário não antecipado detectado (SM travou ou chegou em estado ERR). `None` = tick de monitoramento passivo (nenhuma transição de subtarefa ocorreu neste tick). |

In [7]:
import pandas as pd
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\lucas_alves\Workspace\self-adaptive-arm-simulator\3-dejavu\output\arm\antecipated_scenario_dataset")

EXECUCOES = [
    ("(happy_path)_antecipated_scenario_dataset_20260812_235732.csv", "hp"),
    ("(happy_path)_antecipated_scenario_dataset_20260812_235932.csv", "hp"),
    ("(happy_path)_antecipated_scenario_dataset_20260813_000124.csv", "hp"),
    ("(happy_path)_antecipated_scenario_dataset_20260813_000326.csv", "hp"),
    ("(happy_path)_antecipated_scenario_dataset_20260813_000510.csv", "hp"),
    ("(escorrega_no_lift)_antecipated_scenario_dataset_20260814_012219.csv", "not_hp"),
]

frames = []
for exec_id, (filename, tipo) in enumerate(EXECUCOES, start=1):
    frame = pd.read_csv(DATASET_DIR / filename)
    frame.insert(0, "exec", exec_id)
    frame.insert(1, "tipo", tipo)
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Shape: {df.shape[0]} linhas × {df.shape[1]} colunas")
print(f"Execuções: {df['exec'].nunique()}  |  hp: {(df['tipo']=='hp').sum()} rows  |  not_hp: {(df['tipo']=='not_hp').sum()} rows")
df[["exec", "tipo", "step", "gripper_width_cm", "distance_ee_object_cm", "grasp_completed",
    "finger_contacts", "grasp_attempts", "object_lift_height_cm", "distance_object_goal_cm",
    "reward", "is_success", "current_subtask", "active_state", "active_scenario_name", "active_scenario_type", "sat"]]

Shape: 414 linhas × 101 colunas
Execuções: 6  |  hp: 345 rows  |  not_hp: 69 rows


,exec,tipo,step,gripper_width_cm,distance_ee_object_cm,grasp_completed,finger_contacts,grasp_attempts,object_lift_height_cm,distance_object_goal_cm,reward,is_success,current_subtask,active_state,active_scenario_name,active_scenario_type,sat
0,1,hp,1,7,28,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,APPROACH_OBJECT,D,NaN
1,1,hp,2,7,25,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,APPROACH_OBJECT,D,NaN
2,1,hp,3,7,22,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,APPROACH_OBJECT,D,NaN
3,1,hp,4,7,19,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,APPROACH_OBJECT,D,NaN
4,1,hp,5,7,16,0,0,0,0,40,-0.400001,False,APPROACH_OBJECT,S2,APPROACH_OBJECT,D,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,6,not_hp,65,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
410,6,not_hp,66,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
411,6,not_hp,67,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
412,6,not_hp,68,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False


In [8]:
df_lift = df[df["active_scenario_name"] == "LIFT_OBJECT"].copy()

print(f"Shape: {df_lift.shape[0]} linhas × {df_lift.shape[1]} colunas")
print(f"Execuções: {df_lift['exec'].nunique()}  |  hp: {(df_lift['tipo']=='hp').sum()} rows  |  not_hp: {(df_lift['tipo']=='not_hp').sum()} rows")
df_lift[["exec", "tipo", "step", "gripper_width_cm", "distance_ee_object_cm", "grasp_completed",
         "finger_contacts", "grasp_attempts", "object_lift_height_cm", "distance_object_goal_cm",
         "reward", "is_success", "current_subtask", "active_state", "active_scenario_name", "active_scenario_type", "sat"]]

Shape: 89 linhas × 101 colunas
Execuções: 6  |  hp: 35 rows  |  not_hp: 54 rows


,exec,tipo,step,gripper_width_cm,distance_ee_object_cm,grasp_completed,finger_contacts,grasp_attempts,object_lift_height_cm,distance_object_goal_cm,reward,is_success,current_subtask,active_state,active_scenario_name,active_scenario_type,sat
15,1,hp,16,3,0,1,2,1,1,41,-0.412735,False,LIFT_OBJECT,S18,LIFT_OBJECT,D,True
16,1,hp,17,3,0,1,2,1,4,41,-0.412925,False,LIFT_OBJECT,S18,LIFT_OBJECT,D,True
17,1,hp,18,3,0,1,2,1,7,40,-0.415402,False,LIFT_OBJECT,S18,LIFT_OBJECT,D,True
18,1,hp,19,3,0,1,2,1,10,40,-0.417648,False,LIFT_OBJECT,S18,LIFT_OBJECT,D,True
19,1,hp,20,3,0,1,2,1,13,40,-0.422669,False,LIFT_OBJECT,S18,LIFT_OBJECT,D,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,6,not_hp,65,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
410,6,not_hp,66,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
411,6,not_hp,67,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False
412,6,not_hp,68,7,38,0,0,1,0,41,-0.414327,False,PLACE_OBJECT,ERR_19,LIFT_OBJECT,D,False


---
## Diagnóstico execution-level — por que o LIFT falhou?

### Objetivo

Identificar automaticamente quais parâmetros — não modelados no ASM — explicam por que a pós-condição do cenário `LIFT_OBJECT` não foi satisfeita.

---

### Filtro de linhas

**Unidade de análise: execução, não tick.**

Em vez de treinar sobre todas as linhas do cenário, usamos **uma linha por execução** — o primeiro tick em que `active_scenario_name == LIFT_OBJECT` aparece naquela execução.

```
exec 1 (happy_path)  →  step 16  ← snapshot de entrada no LIFT
exec 2 (happy_path)  →  step 16
exec 3 (happy_path)  →  step 16
exec 4 (happy_path)  →  step 16
exec 5 (happy_path)  →  step 16
exec 6 (escorrega)   →  step 16
```

**Por quê o primeiro tick?** É o momento em que o robô acabou de entrar no cenário — antes de qualquer dinâmica de falha ter acontecido. As linhas seguintes (ERR_19, cubo caindo) descrevem o efeito, não a causa.

**Label por execução:** `lift_failed = True` se aquela execução teve alguma linha com `sat == False` no cenário `LIFT_OBJECT`. Caso contrário, `False`.

---

### Filtro de colunas

São aplicados dois filtros sequenciais:

**1. Remove colunas não treináveis**
- Identificadores: `exec`, `episode`, `step`, `tipo`
- Target: `sat`, `lift_failed`
- Strings: `active_state`, `current_subtask`, `active_scenario_name`, etc. (sklearn não aceita)

**2. Remove colunas constantes globalmente** (`nunique == 1`)
- Features com o mesmo valor em todas as execuções não discriminam nada.
- Ex: `objects.object_1.mass = 1.0` em todas as execuções → descartada.

**3. Separa features estáticas de dinâmicas**

| Categoria | Critério | Exemplos | Papel |
|---|---|---|---|
| **Estática** | `std == 0` dentro de cada execução | `lateral_friction`, `mass`, `robot_config` | Candidata a causa — definida antes do episódio |
| **Dinâmica** | Varia tick a tick dentro da execução | `cube_z`, `cube_yaw`, `gripper_width_cm`, velocidades | Consequência da execução |

Apenas as features **estáticas** entram no treinamento.

---

### Por que apenas features estáticas?

Com N=6 execuções e simulações determinísticas (mesma policy RL, mesma semente), **39 features** separam perfeitamente a execução que falhou das que passaram. A maioria é dinâmica — difere porque o atrito baixo afetou o grasp, que afetou as posições e velocidades. São efeitos, não causas.

Uma árvore de decisão, ao encontrar separação perfeita no primeiro split, para — nunca mostrando features que ficaram "atrás na fila". Ao filtrar para features estáticas, eliminamos o ruído dinâmico e a árvore encontra diretamente a causa.

---

### Dívida técnica ⚠️

O filtro de features estáticas é um **atalho válido para este exemplo**, mas não é generalizável.

**O problema:** com poucas execuções homogêneas, qualquer feature que difere entre a execução que falhou e as que passaram vira "separador perfeito" — inclusive features dinâmicas sem relação causal. O filtro estático contorna isso.

**A solução correta:** mais execuções com variação nos parâmetros causais. Com dados diversos (ex: múltiplos valores de friction em execuções que passaram e falharam), features dinâmicas perderiam consistência e a árvore encontraria `lateral_friction` naturalmente, sem nenhum filtro manual.

---

### Reaproveitamento do diagnoser

O `UnanticipatedScenarioDiagnoser` usa `extract_rules()` para percorrer a árvore e extrair regras legíveis em formato `SE condição → resultado`. O mesmo padrão é reaproveitado aqui — a diferença está apenas na construção do dataset de treino (execution-level + features estáticas em vez de row-level + todas as features).

In [15]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.tree import _tree
import numpy as np

SCENARIO = "LIFT_OBJECT"

# ── 1. Dataset execution-level (1 linha por execução) ─────────────────────────
lift_rows  = df[df["active_scenario_name"] == SCENARIO].copy()

exec_label = (
    lift_rows.groupby("exec")["sat"]
    .apply(lambda s: s.eq(False).any())
    .rename("lift_failed")
)
snapshot = lift_rows.groupby("exec").first()
exec_ds  = snapshot.join(exec_label)

DROP    = ["lift_failed", "sat", "exec", "episode", "step", "tipo"]
X_raw   = exec_ds.drop(columns=[c for c in DROP if c in exec_ds.columns])
X_num   = X_raw.select_dtypes(include=[np.number, bool])
X_all   = X_num.loc[:, X_num.nunique() > 1]   # remove constantes globais
y_exec  = exec_ds["lift_failed"]

# ── 2. Identificar features ESTÁTICAS ─────────────────────────────────────────
#    Estática = std == 0 dentro de cada execução ao longo de TODAS as linhas do cenário
#    Dinâmica = varia tick a tick (posição, velocidade, ângulos do robô, etc.)
static_cols = [
    col for col in X_all.columns
    if col in lift_rows.columns
    and (lift_rows.groupby("exec")[col].std(ddof=0).fillna(0) == 0).all()
]

X_static = X_all[static_cols]
print(f"Features estáticas que variam entre execuções ({len(static_cols)}):")
for c in static_cols:
    print(f"  {c}:  falhou={X_static.loc[y_exec==True, c].values}  "
          f"sucesso={X_static.loc[y_exec==False, c].values[:2]}...")

# ── 3. Árvore de decisão sobre features estáticas ─────────────────────────────
#    Agora a árvore só tem candidatos causais — sem ruído dinâmico
clf = DecisionTreeClassifier(random_state=42, max_depth=3)
clf.fit(X_static, y_exec)

print("\n=== Árvore (features estáticas, execution-level) ===")
print(export_text(clf, feature_names=list(X_static.columns)))

# ── 4. extract_rules — mesmo padrão do UnanticipatedScenarioDiagnoser ─────────
def extract_rules(clf, feature_names, class_names=None):
    tree = clf.tree_
    if class_names is None:
        class_names = [str(c) for c in getattr(clf, "classes_", [])]
    rules = []
    def recurse(node, conditions):
        feat_id = tree.feature[node]
        if feat_id != _tree.TREE_UNDEFINED:
            feat = feature_names[feat_id]
            thr  = round(float(tree.threshold[node]), 4)
            recurse(tree.children_left[node],  conditions + [f"{feat} <= {thr}"])
            recurse(tree.children_right[node], conditions + [f"{feat} > {thr}"])
            return
        value = tree.value[node][0]
        total = float(value.sum())
        proba = value / total if total > 0 else np.zeros_like(value)
        pred  = class_names[int(np.argmax(value))]
        rules.append({"if": conditions, "then": pred,
                      "proba": {class_names[i]: round(float(proba[i]), 4) for i in range(len(proba))},
                      "samples": int(tree.n_node_samples[node])})
    recurse(0, [])
    return rules

rules = extract_rules(clf, list(X_static.columns))

# ── 5. Regra de falha — mesmo padrão do diagnoser ─────────────────────────────
false_rules = [r for r in rules if str(r["then"]) == "True"]   # lift_failed=True
all_conditions = [c for r in false_rules for c in r["if"]]
diagnosed_condition = " AND ".join(all_conditions)

print("=== Condição diagnosticada (reaproveitando padrão do diagnoser) ===")
print(f"  {diagnosed_condition}")
print(f"\n  → Interpretação: quando {diagnosed_condition},")
print(f"    a pós-condição do LIFT não será satisfeita.")

Features estáticas que variam entre execuções (1):
  objects.object_1.lateral_friction:  falhou=[0.14]  sucesso=[0.5 0.5]...

=== Árvore (features estáticas, execution-level) ===
|--- objects.object_1.lateral_friction <= 0.32
|   |--- class: True
|--- objects.object_1.lateral_friction >  0.32
|   |--- class: False

=== Condição diagnosticada (reaproveitando padrão do diagnoser) ===
  objects.object_1.lateral_friction <= 0.32

  → Interpretação: quando objects.object_1.lateral_friction <= 0.32,
    a pós-condição do LIFT não será satisfeita.
